<a href="https://colab.research.google.com/github/aravindchandra/Aravind_INFO5731_Spring2026/blob/main/In_Class_Exercise_Combined_ipynb_(1)_(1)_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Exercise: Sentiment Analysis + Text Classification + Prompt Engineering

**Course:** NLP / Text Mining  
**Total Points:** 30  
**Suggested Time:** about 45 minutes total  
**Dataset:** `sentiment_classification_combined_data.csv`

**Instructions**
- Complete all questions in this notebook.
- Run all cells before submission.
- Keep your answers clear and organized.
- For the prompt engineering part, you do **not** need an API key.


## Part A — Sentiment Analysis with VADER (10 points)
In this part, use **VADER** to compute sentiment scores for the text data.

In [1]:
# Upload the dataset file from your computer (Google Colab)
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
print("Uploaded file name:", filename)


Saving sentiment_classification_combined_data.csv to sentiment_classification_combined_data.csv
Uploaded file name: sentiment_classification_combined_data.csv


### Q1. Load and inspect the dataset (2 points)
Show:
1. the first 5 rows  
2. the shape of the dataset  
3. the column names

In [4]:
# Read the uploaded CSV file

import pandas as pd

# Load the dataset
df = pd.read_csv('sentiment_classification_combined_data.csv')

# Show first 5 rows
print("First 5 rows:")
print(df.head())

# Show shape of dataset
print("\nShape of dataset:")
print(df.shape)

# Show column names
print("\nColumn names:")
print(df.columns)


First 5 rows:
                                                Text TrueLabel  Score
0  I loved this healthcare AI workshop. The examp...  positive      5
1  This lecture was okay, but some parts were con...   neutral      3
2  The notebook was easy to follow and very pract...  positive      5
3  I did not like the explanation of the model re...  negative      1
4  The activity was interesting and helped me und...  positive      4

Shape of dataset:
(24, 3)

Column names:
Index(['Text', 'TrueLabel', 'Score'], dtype='object')


### Q2. Apply VADER to compute sentiment scores (3 points)
- Import VADER
- Download the lexicon if needed
- Create a new column called `compound`
- Show the first 5 rows of `Text` and `compound`

In [5]:
!pip -q install nltk
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [7]:
import pandas as pd
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Load dataset
df = pd.read_csv('sentiment_classification_combined_data.csv')

# Download VADER lexicon (only needed once)
nltk.download('vader_lexicon')

# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Compute compound score
df['compound'] = df['Text'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

# Show first 5 rows (Text + compound)
print(df[['Text', 'compound']].head())

                                                Text  compound
0  I loved this healthcare AI workshop. The examp...    0.8555
1  This lecture was okay, but some parts were con...   -0.2263
2  The notebook was easy to follow and very pract...    0.4404
3  I did not like the explanation of the model re...   -0.2755
4  The activity was interesting and helped me und...    0.4019


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


### Q3. Create a sentiment label from the compound score (3 points)
Create a new column called `PredictedSentiment` using this rule:
- `compound >= 0.05` → `positive`
- `compound <= -0.05` → `negative`
- otherwise → `neutral`

Then show the first 10 rows of `Text`, `compound`, and `PredictedSentiment`.

In [9]:
import pandas as pd
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Load dataset
df = pd.read_csv('sentiment_classification_combined_data.csv')

# Download VADER lexicon
nltk.download('vader_lexicon')

# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Compute compound score
df['compound'] = df['Text'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

# Create PredictedSentiment column
def get_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['PredictedSentiment'] = df['compound'].apply(get_sentiment)

# Show first 10 rows
print(df[['Text', 'compound', 'PredictedSentiment']].head(10))

                                                Text  compound  \
0  I loved this healthcare AI workshop. The examp...    0.8555   
1  This lecture was okay, but some parts were con...   -0.2263   
2  The notebook was easy to follow and very pract...    0.4404   
3  I did not like the explanation of the model re...   -0.2755   
4  The activity was interesting and helped me und...    0.4019   
5  The class was too fast and I could not follow ...    0.0000   
6  The topic is important, but the instructions w...    0.1027   
7  I really enjoyed the examples and the live cod...    0.5563   
8  The dataset was messy and the task felt frustr...   -0.6597   
9  The exercise was manageable and useful for pra...    0.4404   

  PredictedSentiment  
0           positive  
1           negative  
2           positive  
3           negative  
4           positive  
5            neutral  
6           positive  
7           positive  
8           negative  
9           positive  


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


### Q4. Compare review score and sentiment (2 points)
Group by `Score` and compute the **average compound score** for each score value.

In [12]:
import pandas as pd
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Load dataset
df = pd.read_csv('sentiment_classification_combined_data.csv')

# Download VADER lexicon
nltk.download('vader_lexicon')

# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Compute compound score (if not already done)
df['compound'] = df['Text'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

# Group by Score and compute average compound score
avg_compound_by_score = df.groupby('Score')['compound'].mean()

# Display result
print(avg_compound_by_score)

Score
1   -0.507033
2   -0.296125
3   -0.150783
4    0.478340
5    0.621167
Name: compound, dtype: float64


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


## Part B — Prompt Engineering (10 points)
This part is based on the **Prompt Engineering** notebook. You do **not** need an API key. Focus on writing better prompts using the concepts of **instruction, context, output format, zero-shot, few-shot, and chain-of-thought**.

In [13]:
# Sample texts for the prompt engineering questions
sample_reviews = [
    "The product arrived late and the quality was poor.",
    "Amazing battery life and elegant design.",
    "It is fine for daily use, nothing special."
]
for i, text in enumerate(sample_reviews, start=1):
    print(f"{i}. {text}")

1. The product arrived late and the quality was poor.
2. Amazing battery life and elegant design.
3. It is fine for daily use, nothing special.


### Q1. Zero-shot classification prompt (3 points)
Write a **zero-shot prompt** that asks an AI model to classify each review as **positive, negative, or neutral**.

Your prompt should include:
- a clear instruction
- the three labels
- a request for a clean output format

Store your prompt in a Python variable called `zero_shot_prompt`, then print it.

In [14]:
# Zero-shot prompt
zero_shot_prompt = """
You are a sentiment analysis assistant.

Task:
Classify the sentiment of the given review into one of the following labels:
- positive
- negative
- neutral

Instructions:
- Read the review carefully.
- Choose only one label that best represents the overall sentiment.

Output Format:
Return only the label (positive, negative, or neutral).
Do not include any explanation or extra text.

Review:
{review}
"""

# Print the prompt
print(zero_shot_prompt)


You are a sentiment analysis assistant.

Task:
Classify the sentiment of the given review into one of the following labels:
- positive
- negative
- neutral

Instructions:
- Read the review carefully.
- Choose only one label that best represents the overall sentiment.

Output Format:
Return only the label (positive, negative, or neutral).
Do not include any explanation or extra text.

Review:
{review}



### Q2. Few-shot prompt improvement (3 points)
Now improve the previous prompt by adding **two examples**.

Requirements:
- Store it in a variable called `few_shot_prompt`
- Include 2 input-output examples
- Ask the model to classify the 3 sample reviews after the examples

In [15]:
# Few-shot prompt
few_shot_prompt = """
You are a sentiment analysis assistant.

Task:
Classify each review into one of the following labels:
- positive
- negative
- neutral

Instructions:
- Read each review carefully.
- Assign exactly one label per review.

Output Format:
Return the results in this format:
Review 1: <label>
Review 2: <label>
Review 3: <label>

Examples:

Review: "I absolutely loved this product! It works perfectly."
Sentiment: positive

Review: "This is the worst purchase I have ever made."
Sentiment: negative

Now classify the following reviews:

Review 1: "The product is okay, nothing special."
Review 2: "Amazing quality and fast delivery!"
Review 3: "I am very disappointed with the performance."
"""

# Print the prompt
print(few_shot_prompt)


You are a sentiment analysis assistant.

Task:
Classify each review into one of the following labels:
- positive
- negative
- neutral

Instructions:
- Read each review carefully.
- Assign exactly one label per review.

Output Format:
Return the results in this format:
Review 1: <label>
Review 2: <label>
Review 3: <label>

Examples:

Review: "I absolutely loved this product! It works perfectly."
Sentiment: positive

Review: "This is the worst purchase I have ever made."
Sentiment: negative

Now classify the following reviews:

Review 1: "The product is okay, nothing special."
Review 2: "Amazing quality and fast delivery!"
Review 3: "I am very disappointed with the performance."



### Q3. Prompt comparison and refinement (4 points)
Create a short table or DataFrame with **three columns**:
- `PromptType`
- `MainFeature`
- `WhyUseful`

Include these three prompt types:
1. Zero-shot
2. Few-shot
3. Chain-of-thought

Then, in **2–3 sentences**, explain which one you would choose for a more difficult text analysis task and why.

In [16]:
import pandas as pd

# Create the DataFrame
data = {
    "PromptType": ["Zero-shot", "Few-shot", "Chain-of-thought"],
    "MainFeature": [
        "No examples provided",
        "Includes a few labeled examples",
        "Encourages step-by-step reasoning"
    ],
    "WhyUseful": [
        "Quick and simple for straightforward tasks",
        "Improves accuracy by showing patterns",
        "Helps solve complex problems with reasoning steps"
    ]
}

df = pd.DataFrame(data)

# Display the table
print(df)

         PromptType                        MainFeature  \
0         Zero-shot               No examples provided   
1          Few-shot    Includes a few labeled examples   
2  Chain-of-thought  Encourages step-by-step reasoning   

                                           WhyUseful  
0         Quick and simple for straightforward tasks  
1              Improves accuracy by showing patterns  
2  Helps solve complex problems with reasoning steps  


## Reflection (optional, no extra points)
Briefly explain the difference between:
- sentiment analysis
- text classification
- prompt engineering

Sentiment analysis, text classification, and prompt engineering are related but distinct concepts in natural language processing. Sentiment analysis is a specific type of text classification that focuses on determining the emotional tone of a text, such as positive, negative, or neutral.

Text classification is a broader task that involves assigning text to predefined categories, such as topics or labels, including but not limited to sentiment.

Prompt engineering, on the other hand, is the process of designing effective inputs or instructions for AI models to improve their performance on tasks like classification, generation, or analysis.